#### Modules

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from copy import deepcopy

In [11]:
# Check GPU availability - supports NVIDIA CUDA and Apple Metal Performance Shaders (M-chips)
print("PyTorch version:", torch.__version__)
print("\n--- GPU Availability Check ---")

gpu_available = False
device_name = None

# Check NVIDIA CUDA
if torch.cuda.is_available():
    gpu_available = True
    device_name = "NVIDIA CUDA"
    print(f"✓ {device_name} is available")
    print(f"  GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}")

# Check Apple Metal Performance Shaders (M-chips)
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    gpu_available = True
    device_name = "Apple Metal (M-chip)"
    print(f"✓ {device_name} is available")
    if torch.backends.mps.is_built():
        print("  MPS backend is properly built")
    else:
        print("  Warning: MPS backend not built correctly")
else:
    print("✗ No GPU detected - training will use CPU (slower)")
    print("  Supported: NVIDIA CUDA or Apple Metal Performance Shaders (M-chips)")

if gpu_available:
    print(f"\n✓ GPU will be used for training ({device_name})")

PyTorch version: 2.6.0+cu124

--- GPU Availability Check ---
✓ NVIDIA CUDA is available
  GPU count: 1
  - GPU 0: NVIDIA GeForce RTX 3070 Ti Laptop GPU

✓ GPU will be used for training (NVIDIA CUDA)


#### Train and testing splits 

Stratify ensures training, validation and testing datasets have the exact same proportion of crack and uncrack as the original dataset

In [4]:
# Load datasets
df_walls = pd.read_pickle('wall_data.pkl')
df_decks = pd.read_pickle('deck_data.pkl')

# Split the Walls DataFrame (15% of total wall data for validation, hence 0.15 / 0.85 ≈ 0.176 for second split)
walls, walls_te = train_test_split(df_walls, test_size=0.2, random_state=42, stratify=df_walls['Label'])
walls_tr, walls_val = train_test_split(walls, test_size=0.176, random_state=42, stratify=walls['Label'])

# Split the Decks DataFrame
decks, decks_te = train_test_split(df_decks, test_size=0.2, random_state=42, stratify=df_decks['Label'])
decks_tr, decks_val = train_test_split(decks, test_size=0.176, random_state=42, stratify=decks['Label'])

#### CNN Architecture

In [5]:
class CNNBinary(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 30 * 30, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = deepcopy(model.state_dict())
            return False

        self.counter += 1
        return self.counter >= self.patience


criterion = nn.BCEWithLogitsLoss()
label_map = {'Non-Cracked': 0.0, 'Cracked': 1.0}

#### Data Generator (Memory Efficient)

Instead of loading all images into memory at once, we use a generator that loads images in batches during training.

In [6]:
# Custom PyTorch dataset (memory efficient)
class CrackDataset(Dataset):
    def __init__(self, dataframe, label_map):
        self.dataframe = dataframe.reset_index(drop=True)
        self.label_map = label_map

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = row['ImageData'].astype(np.float32) / 255.0
        if image.ndim == 2:
            image = np.stack([image] * 3, axis=-1)

        image = np.transpose(image, (2, 0, 1))
        x = torch.from_numpy(image)
        y = torch.tensor(self.label_map[row['Label']], dtype=torch.float32)

        return x, y

#### Training phase

In [ ]:
# Select device - supports NVIDIA CUDA, Apple Metal (M-chips), or CPU fallback
if torch.cuda.is_available():
    device = torch.device("cuda")
    device_type = "NVIDIA CUDA"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    device_type = "Apple Metal (M-chip)"
else:
    device = torch.device("cpu")
    device_type = "CPU"

print(f"Using device: {device} ({device_type})")

model_walls = CNNBinary().to(device)
model_decks = CNNBinary().to(device)

optimizer_walls = optim.Adam(model_walls.parameters(), lr=1e-3)
optimizer_decks = optim.Adam(model_decks.parameters(), lr=1e-3)


def train_model(model, train_loader, val_loader, optimizer, device, epochs=300, patience=5):
    early_stopper = EarlyStopping(patience=patience)
    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": []
    }

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
            loss.backward()
            
            # For MPS backend on Apple Silicon, use synchronize to ensure proper gradient updates
            if device.type == "mps":
                torch.mps.synchronize()
                
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                logits = model(images).squeeze(1)
                loss = criterion(logits, labels)

                val_running_loss += loss.item() * labels.size(0)
                preds = (torch.sigmoid(logits) >= 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, "
              f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")

        if early_stopper(val_loss, model):
            print("Early stopping triggered.")
            break

    if early_stopper.best_state is not None:
        model.load_state_dict(early_stopper.best_state)

    return history

Using device: cuda


In [8]:
# Create data loaders for Walls dataset (memory efficient)
train_ds_walls = CrackDataset(walls_tr, label_map)
val_ds_walls = CrackDataset(walls_val, label_map)
test_ds_walls = CrackDataset(walls_te, label_map)

pin_memory = torch.cuda.is_available()
train_loader_walls = DataLoader(train_ds_walls, batch_size=32, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader_walls = DataLoader(val_ds_walls, batch_size=32, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader_walls = DataLoader(test_ds_walls, batch_size=32, shuffle=False, num_workers=0, pin_memory=pin_memory)

In [9]:
# Create data loaders for Decks dataset (memory efficient)
train_ds_decks = CrackDataset(decks_tr, label_map)
val_ds_decks = CrackDataset(decks_val, label_map)
test_ds_decks = CrackDataset(decks_te, label_map)

pin_memory = torch.cuda.is_available()
train_loader_decks = DataLoader(train_ds_decks, batch_size=32, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader_decks = DataLoader(val_ds_decks, batch_size=32, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader_decks = DataLoader(test_ds_decks, batch_size=32, shuffle=False, num_workers=0, pin_memory=pin_memory)

In [10]:
# Train using data loaders (memory efficient)
history_walls = train_model(
    model_walls,
    train_loader_walls,
    val_loader_walls,
    optimizer_walls,
    device,
    epochs=300,
    patience=5
)

history_decks = train_model(
    model_decks,
    train_loader_decks,
    val_loader_decks,
    optimizer_decks,
    device,
    epochs=300,
    patience=5
)

Epoch 1/300 - train_loss: 0.5444, train_acc: 0.7867, val_loss: 0.5142, val_acc: 0.7910
Epoch 2/300 - train_loss: 0.5277, train_acc: 0.7907, val_loss: 0.5058, val_acc: 0.7910
Epoch 3/300 - train_loss: 0.5188, train_acc: 0.7909, val_loss: 0.4941, val_acc: 0.7910
Epoch 4/300 - train_loss: 0.5005, train_acc: 0.8000, val_loss: 0.4874, val_acc: 0.8068
Epoch 5/300 - train_loss: 0.4705, train_acc: 0.8156, val_loss: 0.4561, val_acc: 0.8175
Epoch 6/300 - train_loss: 0.4355, train_acc: 0.8271, val_loss: 0.4414, val_acc: 0.8254
Epoch 7/300 - train_loss: 0.3958, train_acc: 0.8452, val_loss: 0.4242, val_acc: 0.8345
Epoch 8/300 - train_loss: 0.3679, train_acc: 0.8579, val_loss: 0.4405, val_acc: 0.8321
Epoch 9/300 - train_loss: 0.3301, train_acc: 0.8743, val_loss: 0.4730, val_acc: 0.8341
Epoch 10/300 - train_loss: 0.2819, train_acc: 0.8921, val_loss: 0.4961, val_acc: 0.8191
Epoch 11/300 - train_loss: 0.2441, train_acc: 0.9063, val_loss: 0.5968, val_acc: 0.8329
Epoch 12/300 - train_loss: 0.2152, train_